In [ ]:
import os
import json
import math
import requests
import time
import glob

from shapely.ops import transform
from shapely.geometry import shape, box, mapping, Point
from shapely.ops import unary_union
from pyproj import Transformer
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

# in meters
cell_size = 1000
google_api_key = "YOUR_API_KEY"

with open('./data/maps/manhattan.geojson') as f:
    manhattan = json.load(f)
    
manhattan_geom = [shape(feature["geometry"]) for feature in manhattan["features"]]

In [ ]:
def fetch_places(body):
    url = 'https://places.googleapis.com/v1/places:searchText'
    headers = {
        'X-Goog-FieldMask': 'places.id,nextPageToken',
        'X-Goog-Api-Key': google_api_key,
        'Content-Type': 'application/json'
    }

    cell_places = []
    page_body = dict(body)
    while True:
        resp = requests.post(url, headers=headers, json=page_body)
        if resp.status_code != 200:
            print(f'HTTP {resp.status_code} -', resp.text)
            break
        data = resp.json()
        places = data.get('places', [])
        cell_places.extend(places)
        token = data.get('nextPageToken')
        if token:
            # Request next page using the returned token
            page_body = dict(body)
            page_body['pageToken'] = token
            # short sleep to respect API pagination timing
            time.sleep(2)
            continue
        break
    
    print(f'Found: {len(cell_places)}')
    return out

# Split manhattan into grid cells

In [ ]:
# Create unified Manhattan geometry from the loaded geojson 'manhattan'
features = manhattan.get('features')
if features is not None:
    polys = [shape(feat['geometry']) for feat in features]
    manh = unary_union(polys)
else:
    manh = shape(manhattan['geometry'])

# Try to use pyproj for accurate meter projection; fall back to approximate projection if unavailable
try:
    to_m = Transformer.from_crs('EPSG:4326', 'EPSG:3857', always_xy=True).transform
    to_wgs = Transformer.from_crs('EPSG:3857', 'EPSG:4326', always_xy=True).transform
    manh_m = transform(to_m, manh)
    use_pyproj = True
except Exception:
    # Approximate conversion using local meters-per-degree at centroid latitude
    latc = manh.centroid.y
    meters_per_deg_lat = 111132.954 - 559.822 * math.cos(2*math.radians(latc)) + 1.175 * math.cos(4*math.radians(latc))
    meters_per_deg_lon = (111412.84 * math.cos(math.radians(latc)) - 93.5 * math.cos(3*math.radians(latc)))
    def project(lon, lat, z=None):
        x = lon * meters_per_deg_lon
        y = lat * meters_per_deg_lat
        return (x, y)
    def project_back(x, y, z=None):
        lon = x / meters_per_deg_lon
        lat = y / meters_per_deg_lat
        return (lon, lat)
    manh_m = transform(project, manh)
    use_pyproj = False

# Grid parameters (meters)
minx, miny, maxx, maxy = manh_m.bounds
xmin = int(minx // cell_size) * cell_size
ymin = int(miny // cell_size) * cell_size
xmax = int(maxx // cell_size + 1) * cell_size
ymax = int(maxy // cell_size + 1) * cell_size

# Keep cells that have a 'good' overlap with Manhattan. Threshold is fraction of cell area (e.g., 0.25 => 25%)
overlap_threshold = 0.05
cells = []
for x in range(xmin, xmax, cell_size):
    for y in range(ymin, ymax, cell_size):
        c = box(x, y, x + cell_size, y + cell_size)
        inter = c.intersection(manh_m)
        if inter.is_empty:
            continue
        if (inter.area / c.area) >= overlap_threshold:
            cells.append(c)

# Transform kept cells back to WGS84 (lon/lat)
if use_pyproj:
    cells_wgs = [transform(to_wgs, c) for c in cells]
else:
    cells_wgs = [transform(project_back, c) for c in cells]

# Write GeoJSON FeatureCollection
features_out = [{'type':'Feature', 'properties':{}, 'geometry': mapping(c)} for c in cells_wgs]
fc = {'type':'FeatureCollection', 'features': features_out}
with open('manhattan_grid.geojson', 'w') as f:
    json.dump(fc, f)

print(f'Wrote manhattan_grid.geojson with {len(features_out)} grid cells (threshold={overlap_threshold})')

In [ ]:
# Load grid
with open('manhattan_grid.geojson') as f:
    grid = json.load(f)

# Search Google Places (SearchText) for 'pizza' inside each grid cell and save results

In [ ]:
all_cells_results = []

for idx, feat in enumerate(grid.get('features', [])):
    
    geom = shape(feat['geometry'])
    minx, miny, maxx, maxy = geom.bounds

    out = fetch_places(
        {
            'textQuery': 'pizza place nyc',
            'includedType': 'pizza_restaurant',
            'pageSize': 20,
            'locationRestriction': {
                'rectangle': {
                    'low': {'latitude': miny, 'longitude': minx},
                    'high': {'latitude': maxy, 'longitude': maxx},
                }
            },
            'strictTypeFiltering': True
        }
    )
    all_cells_results.append(out)
    # save per-cell file
    with open(f'./results_pizza-place-nyc/{idx}.json', 'w') as of:
        json.dump(out, of)
    # small pause between cells to avoid spikes
    time.sleep(0.1)

# Write aggregated results
with open('places_all_cells.json', 'w') as f:
    json.dump(all_cells_results, f)

print(f'Saved results for {len(all_cells_results)} cells to places_all_cells.json')

In [ ]:
# get all unique place IDs

all_place_ids = set()
folders = ['results_pizza-place-nyc', 'results_pizza']

for folder in folders:
    for fpath in glob.glob(f'./{folder}/*.json'):
        with open(fpath) as f:
            data = json.load(f)
            places = data.get('places', [])
            for p in places:
                pid = p.get('id')
                if pid:
                    all_place_ids.add(pid)
                    
print(f'Total unique place IDs found: {len(all_place_ids)}')

# Get details for each place ID from Google Places Details API

In [ ]:

for place_id in all_place_ids:
    fout = f'./data/pizza/{place_id}.json'
    
    if os.path.exists(fout):
        print(f'Place details for {place_id} already exist, skipping.')
        continue
    
    url = f'https://places.googleapis.com/v1/places/{place_id}'
    headers = {
        'X-Goog-FieldMask': 'placeId,name,rating,userRatingCount,location',
        'X-Goog-Api-Key': google_api_key,
    }
    resp = requests.get(url, headers=headers)
    if resp.status_code != 200:
        print(f'HTTP {resp.status_code} -', resp.text)
        continue
    # parse json bytes

    data = json.loads(resp.content.decode('utf-8'))
    
    with open(fout, 'w') as of:
        json.dump(data, of)
    print(f'Saved place details for {place_id} to {fout}')

# Plot data

In [ ]:
# The bounding box for Manhattan to plot
area = dict(
    west=  -74.11,
    east=  -73.84,
    south= 40.69,
    north= 40.875
)

In [ ]:
# load all pizza place details

data = []

for fpath in glob.glob('./data/pizza/*.json'):
    with open(fpath) as f:
        row = json.load(f)
        data.append({
            'place_id': os.path.basename(fpath).replace('.json',''),
            'name': row['displayName']['text'],
            'latitude': row['location']['latitude'],
            'longitude': row['location']['longitude'],
            'rating': row.get('rating'),
            'ratings_count': row.get('userRatingCount'),
        })
        
data = pd.DataFrame(data)
data = data[~pd.isna(data['rating'])].reset_index(drop=True)
data = data.sort_values(by='rating', ascending=True)

# some extra filtering
data = data[data['latitude']< 40.87]
data = data[data['name']!= 'Tasty Fried Chicken']

# remove any not in Manhattan polygon. use the manhattan variable defined earlier

remove = []
for idx, row in data.iterrows():
    point = Point(row['longitude'], row['latitude'])
    if any([area.contains(point) for area in manhattan_geom]):
        continue
    remove.append(idx)

data = data.drop(index=remove).reset_index(drop=True)
print(f"Number of pizza places in Manhattan: {len(data)}")
data.head()


## Plot pizza places on map

In [ ]:

fig = px.scatter_mapbox(
    data,
    lat="latitude",
    lon="longitude",
    color="rating",
    color_continuous_scale=px.colors.sequential.thermal,
    zoom=11,
    mapbox_style="carto-positron",
)
fig.update_traces(marker=dict(size=8, opacity=0.9))
fig.update_layout(
    mapbox=dict(
        bounds=area,
    ),
    margin=dict(l=0, r=0, t=0, b=0),
    coloraxis_colorbar=dict(
        lenmode='fraction',
        yanchor='top',
        thickness=30,
        orientation='h',
        len=0.3,
        y=0.15,
        x=0.85,
        ticks='outside',
        title='Average Rating<br>(1 - 5 stars)',  # ← add line break in title
        title_side='top',
        title_font_size=18,
        xpad=30,
        outlinewidth=1,
        outlinecolor='black',
    )
)

fig.show(renderer='png', width=900, height=900, scale=2)

## Plot histogram of ratings

In [ ]:
# Fixed bins: 2.0 to 5.0 in steps of 0.2
bin_edges = np.arange(2.0, 5.2, 0.2)          # 2.0, 2.2, ..., 5.0
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

# Count occurrences in each bin
counts, _ = np.histogram(data["rating"], bins=bin_edges)

# Normalize for color mapping (2→0, 5→1)
norm = (bin_centers - 2.0) / (5.0 - 2.0)
colors = px.colors.sample_colorscale(px.colors.sequential.thermal, norm)

fig_bar = go.Figure()

fig_bar.add_trace(
    go.Bar(
        x=bin_centers,
        y=counts,
        width=0.2,                        # slightly narrower than bin size for separation
        marker_color=colors,
        marker_line_color='rgba(0,0,0,0.4)',
        marker_line_width=0.8,
        opacity=0.95,
        hovertemplate='Rating: %{x:.1f}–%{x+.1f}<br>Count: %{y}<extra></extra>'
    )
)

fig_bar.update_layout(
    plot_bgcolor='white',
    # have a bit of grid lines for y axis
    yaxis=dict(
        gridcolor='lightgrey',
        gridwidth=1,
    ),
    # title="Distribution of Average Ratings (2–5 stars, 0.2 increments)",
    xaxis_title="Average Rating (out of 5 stars)",
    yaxis_title="Number of Locations",
    bargap=0,                        # small gap; set to 0 for no separation
    xaxis=dict(
        tickmode='array',
        tickvals=np.arange(2.0, 5.1, 0.5),  # ticks every 0.5 for readability
        range=[1.9, 5.1]
    ),
    height=500,
    width=800,
    margin=dict(l=60, r=20, t=80, b=60),
    
    # Colorbar reflecting the 2–5 range
    coloraxis=dict(
        colorscale=px.colors.sequential.thermal,
        colorbar=dict(
            title="Rating",
            title_font_size=16,
            lenmode='fraction',
            len=0.5,
            thickness=20,
            yanchor='middle',
            y=0.5,
            xanchor='right',
            x=1.02,
            ticks='outside',
            title_side='right',
            xpad=15
        ),
        cmin=2.0,
        cmax=5.0
    )
)

# increase font size for axes
fig_bar.update_layout(
    font=dict(
        size=16
    )
)

# have transparent background for saving
fig_bar.update_layout(
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)'
)

fig_bar.show(renderer='png', width=900, height=250)

## Plot density map

In [ ]:
# plot a heatmap of the pizza places using plotly

fig = px.density_mapbox(
    data,
    lat="latitude",
    lon="longitude",
    radius=10,
    center=dict(lat=40.730610, lon=-73.935242),
    zoom=11,
    mapbox_style="carto-positron",
    color_continuous_scale=px.colors.sequential.Sunset,
)

fig.update_layout(
    mapbox=dict(
        bounds=area
    ),
    margin=dict(l=0, r=0, t=0, b=0),
    coloraxis_colorbar=dict(
        lenmode='fraction',
        yanchor='top',
        thickness=30,
        
        orientation='h',
        
        len=0.3,
        y=0.15,
        x=0.85,
        ticks='outside',
        title='Density',
        title_side='top',
        title_font_size=18,
        xpad=30,
        outlinewidth=1,
        outlinecolor='black',
    )
)

fig.show(renderer='png', width=900, height=900, scale=2)